# Introduction to 3DSun

3DSun is a Python package for generating and visualizing 3D meshes of solar features, including the solar surface, magnetic loops, and other solar phenomena.

This notebook provides a basic introduction to the library and demonstrates its core functionality.


## Installation

If you haven't installed the package yet, you can do so by running:

```bash
pip install -e
```

from the repository root directory.


## Basic Usage: Creating a Spherical Mesh

Let's start by importing the necessary modules:


In [ ]:
import numpy as np
import plotly.graph_objects as go
from src import spherical_mesh, visualize_mesh


### Creating a Simple Sphere

First, let's create a simple sphere by defining points on a grid of longitudes and latitudes, all at a constant distance from the center:


In [ ]:
# Create a grid of longitude and latitude points
lon_steps, lat_steps = 20, 15
longitudes = np.linspace(-180, 180, lon_steps)
latitudes = np.linspace(-90, 90, lat_steps)

# Create a meshgrid for all combinations
lon_grid, lat_grid = np.meshgrid(longitudes, latitudes)

# Flatten the grids
lons = lon_grid.flatten()
lats = lat_grid.flatten()

# Create constant distances (perfect sphere)
distances = np.ones_like(lons) * 10.0  # Radius of 10 units

# Combine into points array
points = np.column_stack((lons, lats, distances))

# Generate mesh
vertices, triangles = spherical_mesh(points)

# Visualize
fig = visualize_mesh(vertices, triangles, "Simple Sphere Example", return_fig=True)
fig.show()


### Creating a Bumpy Sphere

Now, let's create a more interesting "bumpy" sphere by varying the distance from the center based on position:


In [ ]:
# Create a grid of longitude and latitude points
lon_steps, lat_steps = 30, 20
longitudes = np.linspace(-180, 180, lon_steps)
latitudes = np.linspace(-90, 90, lat_steps)

# Create a meshgrid for all combinations
lon_grid, lat_grid = np.meshgrid(longitudes, latitudes)

# Flatten the grids
lons = lon_grid.flatten()
lats = lat_grid.flatten()

# Create distances with a simple pattern (a bumpy sphere)
base_radius = 10.0
distances = base_radius + np.sin(np.radians(lons * 2)) * np.cos(np.radians(lats * 2)) * 1.5

# Combine into points array
points = np.column_stack((lons, lats, distances))

# Generate mesh
vertices, triangles = spherical_mesh(points)

# Visualize
fig = visualize_mesh(vertices, triangles, "Bumpy Sphere Example", return_fig=True)
fig.show()


## Advanced Usage: Creating Parametric Tubes (Solar Loops)

3DSun can also create parametric tubes, which are useful for modeling solar loops, prominences, and other structures. Let's import the necessary functions:


In [ ]:
from src.parametric_tube import custom_curve_tube_mesh, loop_with_twist, spiral_curve


### Creating a Simple Loop


In [ ]:
# Create a loop with twist
vertices_loop, triangles_loop = custom_curve_tube_mesh(
    sun_position=(90, -45),  # Position on the sun (longitude, latitude)
    orientation=(0, 0),      # Orientation angles
    base_radius=0.15,        # Base tube thickness
    curve_func=loop_with_twist,
    curve_params={'height': 0.4, 'width': 4, 'twist': 2},
    radius_func=lambda t: 0.15  # Constant radius
)

# Visualize
fig = visualize_mesh(vertices_loop, triangles_loop, "Simple Loop Example", return_fig=True)
fig.show()


### Creating a Spiral (Magnetic Flux Tube)


In [ ]:
# Create a spiral
vertices_spiral, triangles_spiral = custom_curve_tube_mesh(
    sun_position=(180, 30),  # Position on the sun (longitude, latitude)
    orientation=(90, 0),     # Orientation angles
    base_radius=0.1,         # Base tube thickness
    curve_func=spiral_curve,
    curve_params={'height': 3.0, 'radius': 0.2, 'turns': 2, 'taper': 1.1},
    radius_func=lambda t: 0.1 * (1 + t)  # Increasing radius
)

# Visualize
fig = visualize_mesh(vertices_spiral, triangles_spiral, "Spiral Example", return_fig=True)
fig.show()


## Combining Meshes

One of the powerful features of 3DSun is the ability to combine multiple meshes into a single model. Let's create a simple solar model by combining a sphere with some loops:


In [ ]:
from src.generate_3d_mesh import combine_meshes

# Create a simple sphere for the solar surface
lon_steps, lat_steps = 30, 20
longitudes = np.linspace(-180, 180, lon_steps)
latitudes = np.linspace(-90, 90, lat_steps)
lon_grid, lat_grid = np.meshgrid(longitudes, latitudes)
lons = lon_grid.flatten()
lats = lat_grid.flatten()
base_radius = 10.0
distances = base_radius + np.sin(np.radians(lons)) * np.cos(np.radians(lats)) * 0.5
points = np.column_stack((lons, lats, distances))
vertices_sphere, triangles_sphere = spherical_mesh(points)

# Create a loop
vertices_loop1, triangles_loop1 = custom_curve_tube_mesh(
    sun_position=(45, 30),
    orientation=(0, 0),
    base_radius=0.15,
    curve_func=loop_with_twist,
    curve_params={'height': 1.0, 'width': 5, 'twist': 1.5},
    radius_func=lambda t: 0.15
)

# Create another loop
vertices_loop2, triangles_loop2 = custom_curve_tube_mesh(
    sun_position=(-60, -20),
    orientation=(0, 0),
    base_radius=0.2,
    curve_func=loop_with_twist,
    curve_params={'height': 0.8, 'width': 4, 'twist': 1.0},
    radius_func=lambda t: 0.2
)

# Combine all meshes
vertices_combined, triangles_combined = combine_meshes(
    vertices_sphere, triangles_sphere,
    [
        (vertices_loop1, triangles_loop1),
        (vertices_loop2, triangles_loop2)
    ]
)

# Visualize the combined mesh
fig = visualize_mesh(vertices_combined, triangles_combined, "Combined Solar Model", return_fig=True)
fig.show()


## Using Radius Functions

3DSun includes several radius functions for modeling different physical phenomena in solar tubes. Let's import and use some of them:


In [ ]:
from src.radius_functions import magnetic_flux_radius, kink_instability_radius, wavy_radius, composite_radius

# Create a loop with magnetic flux expansion
vertices_magflux, triangles_magflux = custom_curve_tube_mesh(
    sun_position=(120, 45),
    orientation=(0, 0),
    base_radius=0.1,
    curve_func=loop_with_twist,
    curve_params={'height': 1.0, 'width': 5, 'twist': 0.5},
    radius_func=lambda t: magnetic_flux_radius(t, expansion_factor=3.0, corona_start=0.3)
)

# Visualize
fig = visualize_mesh(vertices_magflux, triangles_magflux, "Magnetic Flux Expansion Example", return_fig=True)
fig.show()


### Composite Radius Function

You can also combine multiple radius functions to model complex phenomena:


In [ ]:
# Create a loop with composite radius function
vertices_composite, triangles_composite = custom_curve_tube_mesh(
    sun_position=(-120, -45),
    orientation=(0, 0),
    base_radius=0.1,
    curve_func=loop_with_twist,
    curve_params={'height': 1.2, 'width': 6, 'twist': 1.0},
    radius_func=lambda t: composite_radius(
        t,
        radius_funcs=[magnetic_flux_radius, wavy_radius],
        combination_method='add',
        magnetic_flux_radius={'expansion_factor': 2.0, 'corona_start': 0.4},
        wavy_radius={'wavelength': 8, 'amplitude': 0.05}
    )
)

# Visualize
fig = visualize_mesh(vertices_composite, triangles_composite, "Composite Radius Function Example", return_fig=True)
fig.show()


## Next Steps

This notebook has introduced the basic functionality of 3DSun. For more advanced examples, check out the other notebooks in this directory:

- `data_mesh.ipynb`: Working with real solar data to create meshes
- `harmonics.ipynb`: Using spherical harmonics for solar modeling
- `toroidal.ipynb`: Advanced examples of toroidal structures

You can also explore the example scripts in the `examples/` directory, particularly `sun_model.py` which demonstrates how to create a comprehensive solar model.